In [2]:
import pandas as pd

# Load the Excel workbook
file_path = 'cap_builder.xlsx'
excel_data = pd.ExcelFile(file_path)

# Read the sheets into DataFrames
starting_cap_df = excel_data.parse('starting_cap')
prop_by_manuf_df = excel_data.parse('prop_by_manuf')

In [3]:
#ensure the vaccines are not unique between both, and we can compare/munge
# Extract the unique entries in the 'vaccine' column from both sheets
unique_vaccines_starting_cap = set(starting_cap_df['Vaccine'].unique())
unique_vaccines_prop_by_manuf = set(prop_by_manuf_df['Vaccine'].unique())

# Find the unique entries in each dataset
unique_to_starting_cap = unique_vaccines_starting_cap - unique_vaccines_prop_by_manuf
unique_to_prop_by_manuf = unique_vaccines_prop_by_manuf - unique_vaccines_starting_cap

unique_to_starting_cap, unique_to_prop_by_manuf

(set(), set())

In [4]:
# Merging the two DataFrames on the 'Vaccine' column
merged_df = pd.merge(prop_by_manuf_df, starting_cap_df, how='left', left_on='Vaccine', right_on='Vaccine')

# Applying the proportion to the predicted capacity
merged_df['Adjusted Capacity'] = merged_df['Proportion'] * merged_df['Predicted Capacity']

merged_df.head()


,Manufacturer,Proportion,Vaccine,Predicted Capacity,Adjusted Capacity
0,Biological E. Limited,0.352163,Td,9.803176e+07,3.452312e+07
1,Serum Institute of India Pvt. Ltd.,0.318738,Td,9.803176e+07,3.124647e+07
2,Bul Bio - National Center of Infectious and Pa...,0.192156,Td,9.803176e+07,1.883738e+07
3,PT Bio Farma (Persero),0.136943,Td,9.803176e+07,1.342479e+07
4,Serum Institute of India Pvt. Ltd.,0.525126,HepB,6.977251e+06,3.663934e+06


In [9]:
merged_df.to_excel('producer_portfolio_capacity.xlsx', index=False)

In [5]:
# Aggregating the adjusted capacity by manufacturer
aggregated_capacity_by_manufacturer = merged_df.groupby('Manufacturer')['Adjusted Capacity'].sum().reset_index()

aggregated_capacity_by_manufacturer


,Manufacturer,Adjusted Capacity
0,AJ Vaccines A/S,7.949549e+06
1,Bharat Biotech,1.545015e+07
2,Bilthoven Biologicals,1.242087e+07
3,Biological E. Limited,7.490097e+07
4,Bul Bio - National Center of Infectious and Pa...,2.151824e+07
5,China National Biotec Group,2.349240e+06
6,GlaxoSmithKline Biologicals SA,1.441421e+08
7,Haffkine Bio Pharmaceutical Corporation Ltd,1.818317e+07
8,LG Chem Ltd,2.170722e+07
9,Merck Vaccines,9.165936e+06


In [6]:
output_file_path = 'production_capacity_scenarios.xlsx'
aggregated_capacity_by_manufacturer.to_excel(output_file_path, sheet_name='master_capacity', index=False)